# Step 6: Fixed Source, Fixed PSF/Point Source, Stellar M/L + gNFW


In [ ]:
import os
os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

from pathlib import Path
from copy import deepcopy
import importlib
import warnings
warnings.simplefilter('ignore')

import numpy as np
import xarray as xr
import jax
import jax.numpy as jnp
import numpyro
from numpyro import distributions as dist
from numpyro import infer
from numpyro.infer import autoguide
import optax
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from astropy.io import fits

import herculens_import_main as him
importlib.reload(him)
from herculens_import_main import *
from lens_images_extension import LensImageExtension
from herculens.PointSourceModel.point_source_model import PointSourceModel
from herculens.MassModel.mass_model import MassModel
from herculens.MassModel import mass_model_base
from herculens.LightModel.light_model import LightModel
from herculens.Instrument.psf import PSF
from herculens.Instrument.noise import Noise
from jax_lensing_profiles.MassModel.Profiles.CuspyNFW_ellipse_kappa import CuspyNFW_3D_fn
from jax_lensing_profiles.MassModel.Profiles.MGE import MGE

jax.config.update('jax_enable_x64', True)
numpyro.enable_x64()


In [ ]:
class CuspyNFWEllipseKappa(MGE):
    def __init__(self):
        super().__init__(
            CuspyNFW_3D_fn,
            'R_s',
            n_gauss=30,
            n_terms=28,
            sigma_start_mult=1e-4,
            sigma_end_mult=5,
            three_d=True,
        )

mass_model_base.STRING_MAPPING['CUSPY_NFW_ELLIPSE_KAPPA'] = CuspyNFWEllipseKappa

suffix = '_ss=2_full_light'
RESULT_DIR = Path('/mnt/d/lensing/Herculens/Herculensedquasar/WFI2033/result/result_ss=2_full_light_20260326_14')
PRODUCTS_DIR = RESULT_DIR / 'data_products'
DATA_DIR = Path('/mnt/d/lensing/Herculens/Herculensedquasar/Data/WFI2033')
RAW_DATA_PATH = DATA_DIR / 'jw01198-o004_t004_nircam_clear-f115w_i2d.fits'
DATA_SUB_PATH = RESULT_DIR / f'data_minus_lens_light{suffix}.fits'
HMC_MEDIAN_PATH = RESULT_DIR / f'HMC_median_draw{suffix}.nc'
FIXED_FIRST_THREE_PATH = PRODUCTS_DIR / f'fixed_first_three_gaussians{suffix}.npz'

with fits.open(RAW_DATA_PATH, memmap=True) as hdul_raw:
    raw_header = hdul_raw['SCI'].header if 'SCI' in hdul_raw else hdul_raw[0].header
    exposure_time = float(raw_header.get('EXPTIME', raw_header.get('TEXPTIME', raw_header.get('XPOSURE', 1.0))))
pix_scale = float(np.sqrt(raw_header['PIXAR_A2']))

data = np.array(fits.getdata(PRODUCTS_DIR / f'data_bkg_sub{suffix}.fits'), dtype=float)
data_subtracted = np.array(fits.getdata(DATA_SUB_PATH), dtype=float)
rms_file = np.array(fits.getdata(PRODUCTS_DIR / f'rms_with_psf_extra{suffix}.fits'), dtype=float)
mask_out = np.array(fits.getdata(DATA_DIR / 'mask_out_center.fits'), dtype=bool)
fixed_first_three = np.load(FIXED_FIRST_THREE_PATH)
HMC_median = xr.load_dataset(HMC_MEDIAN_PATH).isel(chain=slice(0, 4))
HMC_all_chain_median = HMC_median.median(dim='chain')

ny, nx = mask_out.shape
xc, yc = nx / 2, ny / 2
r = 16
Y, X = np.indices((ny, nx))
mask_out = np.logical_or(mask_out, (X - xc) ** 2 + (Y - yc) ** 2 <= r ** 2)

pixel_grid_shape = 75
source_grid_scale = 0.8
ss_factor = 2
num_chains = 4

G1_MASS_CENTER = (1.556, 1.299)
G2_MASS_CENTER = (2.145, -3.326)
conj_points = jnp.array([
    [1.20212170716053, -0.12271885209256231],
    [0.9053233071260114, 0.5277189685977776],
    [-1.0461673774453952, 1.0081083299749878],
    [-0.1255456241215261, -0.8965524340129204],
])


In [ ]:
fixed_inner_three = [{
    'amp': np.array(fixed_first_three['amp'], dtype=float),
    'sigma': np.array(fixed_first_three['sigma'], dtype=float),
    'e1': np.array(fixed_first_three['e1'], dtype=float),
    'e2': np.array(fixed_first_three['e2'], dtype=float),
    'center_x': np.array(fixed_first_three['center_x'], dtype=float),
    'center_y': np.array(fixed_first_three['center_y'], dtype=float),
}]

outer_two = [{
    'amp': np.array(HMC_all_chain_median['amp_lens'].values, dtype=float)[-2:],
    'sigma': np.array(HMC_all_chain_median['sigma_lens'].values, dtype=float)[-2:],
    'e1': np.array(HMC_all_chain_median['e_lens'].values, dtype=float)[0, -2:],
    'e2': np.array(HMC_all_chain_median['e_lens'].values, dtype=float)[1, -2:],
    'center_x': np.array(HMC_all_chain_median['center_lens'].values, dtype=float)[0, -2:],
    'center_y': np.array(HMC_all_chain_median['center_lens'].values, dtype=float)[1, -2:],
}]

full_lens_light = [{
    k: np.concatenate([np.asarray(fixed_inner_three[0][k]), np.asarray(outer_two[0][k])])
    for k in ('amp', 'sigma', 'e1', 'e2', 'center_x', 'center_y')
}]

fixed_source = [{
    'pixels': np.array(HMC_all_chain_median['pixels_source_grid'].values, dtype=float)
}]

fixed_point_source = [{
    'ra': np.array(HMC_all_chain_median['ra_ps'].values, dtype=float),
    'dec': np.array(HMC_all_chain_median['dec_ps'].values, dtype=float),
    'amp': np.power(10.0, np.array(HMC_all_chain_median['log10_amp_ps'].values, dtype=float)),
}]

fixed_psf = np.array(HMC_all_chain_median['psf_kernel_corrected'].values, dtype=float)
fixed_psf = np.clip(fixed_psf, 0.0, None)
fixed_psf = fixed_psf / fixed_psf.sum()

epl_center = np.array(HMC_all_chain_median['center_1'].values, dtype=float).reshape(2,)
fixed_sis_g1 = [{
    'theta_E': float(np.asarray(HMC_all_chain_median['theta_E_g1'].values).reshape(-1)[0]),
    'center_x': float(G1_MASS_CENTER[0]),
    'center_y': float(G1_MASS_CENTER[1]),
}]
fixed_sis_g2 = [{
    'theta_E': float(np.asarray(HMC_all_chain_median['theta_E_g2'].values).reshape(-1)[0]),
    'center_x': float(G2_MASS_CENTER[0]),
    'center_y': float(G2_MASS_CENTER[1]),
}]

pixel_grid, xgrid, ygrid, x_axis, y_axis, extent, nx, ny = get_pixel_grid(jnp.asarray(data_subtracted), pix_scale)
noise = Noise(nx, ny, exposure_time=exposure_time)
psf_obj = PSF(psf_type='PIXEL', kernel_point_source=fixed_psf)

mass_model_step6 = MassModel([
    'MULTI_GAUSSIAN_ELLIPSE_KAPPA',
    'CUSPY_NFW_ELLIPSE_KAPPA',
    'SHEAR',
    'SIS',
    'SIS',
])
source_light_model = LightModel(
    ['PIXELATED'],
    pixel_adaptive_grid=True,
    pixel_interpol='fast_bilinear',
    kwargs_pixelated={'num_pixels': pixel_grid_shape},
)
point_source_model = PointSourceModel(
    ['IMAGE_POSITIONS'],
    mass_model=mass_model_step6,
    image_plane=deepcopy(pixel_grid),
)
lens_image_step6 = LensImageExtension(
    deepcopy(pixel_grid),
    psf_obj,
    noise_class=noise,
    lens_light_model_class=LightModel(['MULTI_GAUSSIAN_ELLIPSE'], {}),
    lens_mass_model_class=mass_model_step6,
    source_model_class=source_light_model,
    point_source_model_class=point_source_model,
    source_arc_mask=jnp.array(mask_out),
    conjugate_points=conj_points,
    kwargs_numerics={'supersampling_factor': ss_factor},
    source_grid_scale=source_grid_scale,
)


In [ ]:
lens_light_image = np.array(data - data_subtracted, dtype=float)

fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
ax[0].imshow(np.ma.array(data, mask=~mask_out), origin='lower', extent=extent, cmap='twilight', norm='log')
ax[0].set_title('data')
ax[1].imshow(np.ma.array(lens_light_image, mask=~mask_out), origin='lower', extent=extent, cmap='twilight', norm='log')
ax[1].set_title('fixed lens light')
ax[2].imshow(np.ma.array(data_subtracted, mask=~mask_out), origin='lower', extent=extent, cmap='twilight', norm='log')
ax[2].set_title('data - lens light')
plt.tight_layout()
plt.show()


In [ ]:
def build_mass_kwargs_from_params(params):
    amp = jnp.asarray(full_lens_light[0]['amp'])
    stellar = {
        'amp': amp * params['m2l_ratio'] / jnp.sum(amp),
        'sigma': jnp.asarray(full_lens_light[0]['sigma']),
        'e1': jnp.asarray(full_lens_light[0]['e1']),
        'e2': jnp.asarray(full_lens_light[0]['e2']),
        'center_x': jnp.asarray(full_lens_light[0]['center_x']),
        'center_y': jnp.asarray(full_lens_light[0]['center_y']),
    }
    halo = {
        'R_s': 5.0,
        'gamma': float(np.asarray(params['gammain_halo']).reshape(-1)[0]),
        'kappa_s': float(np.asarray(params['kappa_s_halo']).reshape(-1)[0]),
        'e1': 1e-4,
        'e2': -1e-4,
        'center_x': float(epl_center[0]),
        'center_y': float(epl_center[1]),
    }
    shear = {
        'gamma1': float(np.asarray(params['gamma_sheer_halo']).reshape(-1)[0]),
        'gamma2': float(np.asarray(params['gamma_sheer_halo']).reshape(-1)[1]),
        'ra_0': float(epl_center[0]),
        'dec_0': float(epl_center[1]),
    }
    return [stellar, halo, shear] + fixed_sis_g1 + fixed_sis_g2


def model_step6(data_subtracted):
    m2l_ratio = numpyro.sample('m2l_ratio', dist.Uniform(0.0, 14.0))
    gnfw_shear = GNFW_w_shear(
        'Lens mass',
        'halo',
        gamma_in_up=1.01,
        gamma_in_low=0.99,
        Rs_value=5.0,
        sph=True,
        center_x=float(epl_center[0]),
        center_y=float(epl_center[1]),
        gamma_sheer_low=-0.5,
        gamma_sheer_high=0.5,
    )
    amp = jnp.asarray(full_lens_light[0]['amp'])
    mass_from_light = [{
        'amp': amp * m2l_ratio / jnp.sum(amp),
        'sigma': jnp.asarray(full_lens_light[0]['sigma']),
        'e1': jnp.asarray(full_lens_light[0]['e1']),
        'e2': jnp.asarray(full_lens_light[0]['e2']),
        'center_x': jnp.asarray(full_lens_light[0]['center_x']),
        'center_y': jnp.asarray(full_lens_light[0]['center_y']),
    }]
    kwargs_lens = mass_from_light + gnfw_shear + fixed_sis_g1 + fixed_sis_g2
    model_image = lens_image_step6.model(
        kwargs_lens=kwargs_lens,
        kwargs_source=fixed_source,
        kwargs_lens_light=[],
        kwargs_point_source=fixed_point_source,
        source_add=True,
        lens_light_add=False,
        point_source_add=True,
        psf_kernel=fixed_psf,
    )
    numpyro.deterministic('model_image', model_image)

    with numpyro.plate(f'data - [{int(mask_out.sum())}]', int(mask_out.sum())):
        numpyro.sample('obs', dist.Normal(model_image[mask_out], jnp.asarray(rms_file)[mask_out]), obs=jnp.asarray(data_subtracted)[mask_out])


max_iterations = 5000
scheduler = optax.exponential_decay(init_value=5e-3, transition_steps=300, decay_rate=0.99)
optim = optax.adabelief(learning_rate=scheduler)
loss = infer.TraceMeanField_ELBO()

rng = jax.random.PRNGKey(1234)
keys = jax.random.split(rng, num_chains)
step6_results = []
step6_medians = []

for i in range(num_chains):
    guide = autoguide.AutoDiagonalNormal(model_step6, init_loc_fn=infer.init_to_median(num_samples=20), init_scale=0.01)
    svi = infer.SVI(model_step6, guide, optim, loss)
    result = svi.run(keys[i], max_iterations, data_subtracted, progress_bar=True, stable_update=True)
    step6_results.append(result)
    step6_medians.append(guide.median(result.params))


In [ ]:
plt.figure(figsize=(10, 4))
for i, result in enumerate(step6_results):
    plt.plot(result.losses, alpha=0.6, label=f'chain {i}')
plt.yscale('log')
plt.xlabel('iteration')
plt.ylabel('ELBO loss')
plt.legend()
plt.show()

for i, params in enumerate(step6_medians):
    kwargs_lens = build_mass_kwargs_from_params(params)
    model_image = np.array(
        lens_image_step6.model(
            kwargs_lens=kwargs_lens,
            kwargs_source=fixed_source,
            kwargs_lens_light=[],
            kwargs_point_source=fixed_point_source,
            source_add=True,
            lens_light_add=False,
            point_source_add=True,
            psf_kernel=fixed_psf,
        )
    )
    residual = (data_subtracted - model_image) / rms_file

    fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
    fig.suptitle(
        f'chain {i} | M/L = {float(np.asarray(params["m2l_ratio"]).reshape(-1)[0]):.3f} | '
        f'kappa_s = {float(np.asarray(params["kappa_s_halo"]).reshape(-1)[0]):.4f} | '
        f'gamma = {float(np.asarray(params["gammain_halo"]).reshape(-1)[0]):.4f}',
        y=1.02,
    )

    ax[0].imshow(np.ma.array(data_subtracted, mask=~mask_out), origin='lower', extent=extent, cmap='twilight', norm='log')
    ax[0].set_title('data - lens light')

    ax[1].imshow(np.ma.array(model_image, mask=~mask_out), origin='lower', extent=extent, cmap='twilight', norm='log')
    ax[1].set_title('model')

    im = ax[2].imshow(np.ma.array(residual, mask=~mask_out), origin='lower', extent=extent, cmap='bwr', vmin=-3, vmax=3)
    ax[2].set_title('residual / rms')
    plt.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04)

    ax[3].imshow(np.array(fixed_source[0]['pixels']), origin='lower', cmap='twilight')
    ax[3].set_title('fixed source')

    plt.tight_layout()
    plt.show()
